# "THE PRICE IS RIGHT" Capstone Project

This week - build a model that predicts how much something costs from a description, based on a scrape of Amazon data

A model that can estimate how much something costs, from its description.

# Order of play

DAY 1: Data Curation  
DAY 2: Data Pre-processing  
DAY 3: Evaluation, Baselines, Traditional ML  
DAY 4: Deep Learning and LLMs  
DAY 5: Fine-tuning a Frontier Model

## DAY 2: Data Pre-processing

Today we'll rewrite the products into a standard format.  
LLMs are great at this!


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business value of Data Pre-processing / Re-writing</h2>
            <span style="color:#181;">LLMs have made it simple to do something that was considered impossible only a few years ago.
            This approach can be applied to almost any business vertical, and it's similar to the advanced techniques
            we used on Week 5.</span>
        </td>
    </tr>
</table>


In [1]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

/Users/tayjiasheng/AI Projects/llm_engineering/.venv/lib/python3.12/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.3.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


True

# The next cell is where you choose Dataset

Use `LITE_MODE = True` for the free, fast version with training data size of 20,000

USe `LITE_MODE =  False` for the powerful, full version with training data size of 800,000

## For this lab

You can skip altogether and load the dataset from HuggingFace: $0

You can run pre-processing for the lite dataset: under $1

You can run pre-processing for the full dataset: $30


In [ ]:
LITE_MODE = True

In [ ]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

In [ ]:
items[
    2
].id  # initially its None, after we run the code block below and run this again, the item now has an id

In [ ]:
# Give every item an id (based on its index)

for index, item in enumerate(items):
    item.id = index

In [ ]:
SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

# why don't ask the model to respond in json? - yes, you can use structured outputs (used in week 8), however not used here because we are working on such a large dataset -> expensive (json has a lot of stuff to it.)

In [ ]:
print(
    items[0].full
)  # this full description is very messy, a lot of filler / noise as well. Should use an LLM that is best suited for training.

In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full},
]
response = completion(
    messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low"
)

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(
    f"Output tokens: {response.usage.completion_tokens}"
)  # the output tokens includes REASONING tokens as well, not ONLY the output.
print(f"Cost: {response._hidden_params['response_cost'] * 100:.3f} cents")


In [ ]:
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": items[0].full},
]
response = completion(
    messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434"
)  # to be cost-conscious, use a free local model, but quality suffers compared to the oss 20B.
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost'] * 100:.3f} cents")


In [ ]:
MODEL = "openai/gpt-oss-20b"


In [ ]:
def make_jsonl(item):
    # we are making a single line of json, for each request we want to make to the groq batch mode api (openai api has a similar batch mode as well)
    body = {
        "model": MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": item.full},
        ],
        "reasoning_effort": "low",
    }
    line = {
        "custom_id": str(
            item.id
        ),  # each line NEEDS an ID number to identify each request you are sending to the batch mode, since later you are getting the responses back for each line as well.
        "method": "POST",
        "url": "/v1/chat/completions",
        "body": body,
    }
    return json.dumps(line)

In [ ]:
items[0]

In [ ]:
make_jsonl(items[0])

In [ ]:
def make_file(start, end, filename):
    # make a jsonl file, by iternating each item and creating a json row in the jsonl file
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

### 4 steps to use Batch Mode


In [ ]:
# Step 1 of batch mode: CREATE THE FILE (groq.files.create)
with open("jsonl/0_1000.jsonl", "rb", encoding="utf-8") as f:
    response = groq.files.create(file=f, purpose="batch")  # create the file
response

In [ ]:
file_id = response.id
file_id

In [ ]:
# 2. USE THAT FILE TO CREATE THE BATCH (groq.batches.create)
response = groq.batches.create(
    completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id
)  # WE give it a completion_window of "24h", to get half price
response

In [ ]:
# 3. RETRIEVE THE BATCH (even though we gave it 24h to do it, we run this code block next and it's already done!) (groq.batches.retrieve)
result = groq.batches.retrieve(response.id)
result

In [ ]:
# 4. COLLECT THE RESULTS (which is a jsonl file - we cannot rely on the order of the json, because groq has a farm of processes that handles our requests in any order - hence we need that ID to match the response to the correct json in the jsonl file in step 1) (groq.batches.content)
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

In [ ]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])  # take the id
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[
            id
        ].summary = summary  # attach the summary to the correct item in the dataset


In [ ]:
print(items[0].full)

In [ ]:
print(items[1000].summary)

## I've put exactly this logic into a Batch class

- Divides items into groups of 1,000
- Kicks off batches for each
- Allows us to monitor and collect the results when complete

## COSTS

Using Groq, for me - this cost under $1 for the Lite dataset and under $30 for the big dataset

But you don't need to pay anything! In the next lab, you can load my pre-processed results


In [ ]:
Batch.create(items, LITE_MODE)

In [ ]:
Batch.run()

In [ ]:
Batch.fetch()

In [ ]:
for index, item in enumerate(items):
    # this just makes sures that EVERY ITEM should have an item.summary after the Batch.fetch() is complete. can run Batch.fetch() as many times as required as it will just return the unfinished calls
    if not item.summary:
        print(index)

In [ ]:
print(items[10234].summary)

In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

## Push the final dataset to the hub

If lite mode, we'll only push the lite dataset

If full mode, we'll push both datasets (in case you decide to use lite later)


In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

## And here they are!

https://huggingface.co/datasets/ed-donner/items_lite

https://huggingface.co/datasets/ed-donner/items_full
